# Recs 007: User-facing qualitative evaluation

## Key Goal

Run user-facing qualitative checks on recommendation outputs to assess clarity, relevance, and perceived usefulness beyond offline metrics.


Purpose:
- Run a small, repeatable qualitative review of recommendation quality.
- Keep this separate from metric-heavy proxy notebooks (`recs_004`, `recs_006`).

Flow:
1. Load evaluation prompts.
2. Generate top-K recommendations with current default path (`raw_raw`).
3. Export a review sheet.
4. Fill manual judgments and summarize failure modes.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import pandas as pd

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")

REPO_ROOT = _repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from steam_review_ml.recommender.retrieve import ContentRetriever

EVAL_JSONL = REPO_ROOT / "artifacts" / "recs" / "eval_queries_review_style.jsonl"
OUT_CSV = REPO_ROOT / "artifacts" / "recs" / "user_facing_qual_eval_sheet.csv"
OUT_SUMMARY_CSV = REPO_ROOT / "artifacts" / "recs" / "user_facing_qual_eval_summary.csv"

TOP_K = 10
print("Eval prompts:", EVAL_JSONL)
print("Output sheet:", OUT_CSV)

Eval prompts: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_queries_review_style.jsonl
Output sheet: /home/ryanr/workspace/steam_recommendations/artifacts/recs/user_facing_qual_eval_sheet.csv


In [2]:
def load_jsonl(path: Path) -> list[dict]:
    out: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

queries = load_jsonl(EVAL_JSONL)
retriever = ContentRetriever()
print("Loaded queries:", len(queries))
print("Indexed games:", len(retriever.index_frame))

Loaded queries: 31
Indexed games: 315


In [3]:
# pandas set options make the output more readable
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.5f}'.format)


In [4]:
evaluation_rows: list[dict] = []
for query_row in queries:
    query_id = query_row.get("id", "")
    review_draft_text = query_row.get("review_draft", "")
    expected_theme_list = query_row.get("expected_themes", [])
    avoided_theme_list = query_row.get("avoid_themes", [])

    retrieval_hits = retriever.top_k(review_draft_text, k=TOP_K, structured=False)
    recommended_game_names = retrieval_hits["app_name"].tolist()

    evaluation_rows.append(
        {
            "id": query_id,
            "review_draft": review_draft_text,
            "expected_themes": " | ".join(expected_theme_list),
            "avoid_themes": " | ".join(avoided_theme_list),
            "top1": recommended_game_names[0] if recommended_game_names else "",
            "top3": " | ".join(recommended_game_names[:3]),
            "top10": " | ".join(recommended_game_names),
            "quality_label": "",  # manual: good / mixed / bad
            "failure_tags": "",   # manual: semicolon-separated tags
            "notes": "",          # manual free-text rationale
        }
    )

qualitative_eval_sheet = pd.DataFrame(evaluation_rows)
qualitative_eval_sheet.head(10)

2026-04-16 08:08:23.997236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776341304.013795   22429 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776341304.020355   22429 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776341304.041237   22429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776341304.041271   22429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776341304.041272   22429 computation_placer.cc:177] computation placer alr

,id,review_draft,expected_themes,avoid_themes,top1,top3,top10,quality_label,failure_tags,notes
0,r001,"Combat feels amazing when parries click, and boss fights are memorable. The story is kind of cryptic and sometimes I had no clue where to go next.",challenging melee combat | boss-focused action | dark fantasy tone,easy casual gameplay,Titan Souls,Titan Souls | Sekiro™: Shadows Die Twice | Momodora: Reverie Under the Moonlight,Titan Souls | Sekiro™: Shadows Die Twice | Momodora: Reverie Under the Moonlight | Darksiders III | Salt and Sanctuary | Hollow Knight | Dead Cells | Iconoclasts | Tales of Berseria | Dungreed,,,
1,r002,"Loved the farming loop and decorating my house. Days were relaxing and I could just chill, but the mines got repetitive after a while.",cozy progression | farming/life sim | relaxing pace,combat-heavy grind,Farm Together,Farm Together | Townscaper | House Flipper,Farm Together | Townscaper | House Flipper | A Short Hike | Stardew Valley | Farm Manager 2018 | Staxel | My Time At Portia | The Sims(TM) 3 | Slime Rancher,,,
2,r003,"The squad tactics are the best part: flanking, cover, and risky plays. Missing a 90 percent shot still drives me insane though.",turn-based tactics | squad strategy | positioning,real-time twitch gameplay,Sniper Elite 4,Sniper Elite 4 | XCOM 2 | Tom Clancy's Rainbow Six Siege,Sniper Elite 4 | XCOM 2 | Tom Clancy's Rainbow Six Siege | Steel Division: Normandy 44 | Football Manager 2019 | Takedown: Red Sabre | Due Process | Bomber Crew | Dead by Daylight | Day of Infamy,,,
3,r004,"Characters were great and I cared about their arcs. Side quests actually had consequences, unlike most open-world filler.",narrative RPG | companion writing | choices/consequences,checklist open world,Tales of Berseria,Tales of Berseria | Assassin's Creed Odyssey | DRAGON QUEST HEROES™ II,Tales of Berseria | Assassin's Creed Odyssey | DRAGON QUEST HEROES™ II | Fairy Fencer F Advent Dark Force | The Legend of Heroes: Trails of Cold Steel II | Ni no Kuni™ II: Revenant Kingdom | Pillars of Eternity II: Deadfire | Torment: Tides of Numenera | The Walking Dead | Sword Art Online: Fatal Bullet,,,
4,r005,"Finished it in one weekend. Short, stylish, and atmospheric. I appreciated that it did not overstay its welcome.",short indie | atmospheric experience | concise design,very long runtime,FAR: Lone Sails,FAR: Lone Sails | Grimm's Hollow | Helltaker,FAR: Lone Sails | Grimm's Hollow | Helltaker | There Is No Game: Wrong Dimension | A Short Hike | The Room | Gunpoint | We Were Here Too | The Room Two | Little Nightmares,,,
5,r006,City planning and logistics are deep and satisfying. Traffic management is brutal but in a good way.,city builder | management sim | optimization/logistics,light arcade gameplay,Cities: Skylines,Cities: Skylines | Euro Truck Simulator 2 | Banished,Cities: Skylines | Euro Truck Simulator 2 | Banished | Railway Empire | Urban Empire | Frostpunk | Rise of Industry | American Truck Simulator | Surviving Mars | Steel Division: Normandy 44,,,
6,r007,Gunplay is crisp and movement feels smooth. Matchmaking can be rough but when teams are balanced it is insanely fun.,competitive multiplayer FPS | tight gunplay | high skill ceiling,single-player narrative focus,Due Process,Due Process | Tom Clancy's Rainbow Six Siege | Insurgency: Sandstorm,Due Process | Tom Clancy's Rainbow Six Siege | Insurgency: Sandstorm | For Honor | Day of Infamy | BATTALION 1944 | Totally Accurate Battlegrounds | Takedown: Red Sabre | Umbrella Corps | Hunt: Showdown,,,
7,r008,"Piecing together clues was the highlight, and I liked that the game trusted me to think. Jump scares felt cheap when they showed up.",investigation | mystery solving | deductive gameplay,jump-scare horror,Outlast,Outlast | Phasmophobia | Little Nightmares,Outlast | Phasmophobia | Little Nightmares | Detention | Hunt Down The Freeman | Night in the Woods | There Is No Game: Wrong Dimension | Thief Simulator | Ghost of a Tale | The Henry Stickmin Collectio

In [5]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
qualitative_eval_sheet.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)
print("\nHow to annotate:")
print("1) Open the CSV and fill quality_label with: good / mixed / bad")
print("2) Fill failure_tags with semicolon-separated tags, e.g. genre_mismatch;popularity_drift")
print("3) Add short notes explaining why")
print("4) Save CSV, then run the next cell to summarize")

Wrote: /home/ryanr/workspace/steam_recommendations/artifacts/recs/user_facing_qual_eval_sheet.csv

How to annotate:
1) Open the CSV and fill quality_label with: good / mixed / bad
2) Fill failure_tags with semicolon-separated tags, e.g. genre_mismatch;popularity_drift
3) Add short notes explaining why
4) Save CSV, then run the next cell to summarize


In [6]:
# Re-load manually annotated sheet and summarize.
annotated_sheet = pd.read_csv(OUT_CSV)

quality_label_counts = annotated_sheet["quality_label"].fillna("").value_counts()
print("Quality label counts:")
display(quality_label_counts)

failure_tag_series = (
    annotated_sheet["failure_tags"].fillna("")
    .str.split(";")
    .explode()
    .str.strip()
)
failure_tag_series = failure_tag_series[failure_tag_series != ""]
failure_tag_counts = failure_tag_series.value_counts()
print("\nFailure tag counts:")
display(failure_tag_counts)

summary = pd.DataFrame({
    "metric": ["n_rows", "n_good", "n_mixed", "n_bad"],
    "value": [
        len(annotated_sheet),
        int((annotated_sheet["quality_label"] == "good").sum()),
        int((annotated_sheet["quality_label"] == "mixed").sum()),
        int((annotated_sheet["quality_label"] == "bad").sum()),
    ],
})
summary.to_csv(OUT_SUMMARY_CSV, index=False)
print("\nWrote:", OUT_SUMMARY_CSV)

Quality label counts:


quality_label
    31
Name: count, dtype: int64


Failure tag counts:


Series([], Name: count, dtype: int64)


Wrote: /home/ryanr/workspace/steam_recommendations/artifacts/recs/user_facing_qual_eval_summary.csv


In [7]:
# Optional: summarize manual failure tags
from pathlib import Path
import pandas as pd

try:
    import yaml
except ImportError as e:
    raise ImportError("PyYAML is required for this cell. Install with: pip install pyyaml") from e

FAILURE_TAGS_PATH = REPO_ROOT / 'notebooks' / 'models' / 'query_embeddings' / 'failure_tags.yaml'
with FAILURE_TAGS_PATH.open('r', encoding='utf-8') as f:
    tag_data = yaml.safe_load(f)

reviews = pd.DataFrame(tag_data.get('reviews', []))
if 'failure_tags' not in reviews.columns:
    reviews['failure_tags'] = [[] for _ in range(len(reviews))]
reviews['failure_tags'] = reviews['failure_tags'].apply(lambda x: x if isinstance(x, list) else [])

print('Manual review rows:', len(reviews))
display(reviews[['quality_label']].value_counts().rename('count').to_frame())

tag_counts = (
    reviews[['id', 'quality_label', 'failure_tags']]
    .explode('failure_tags')
    .dropna(subset=['failure_tags'])
    .groupby('failure_tags')
    .size()
    .sort_values(ascending=False)
)
print('Failure tag counts:')
display(tag_counts.rename('count').to_frame())

bad_mixed = reviews[reviews['quality_label'].isin(['bad', 'mixed'])].copy()
print('Bad/mixed rows:', len(bad_mixed))
display(bad_mixed[['id', 'quality_label', 'failure_tags', 'notes']])


Manual review rows: 30


,count
quality_label,
good,26
bad,3
mixed,1


Failure tag counts:


,count
failure_tags,
bad_top_1,5
flipped_rank,2


Bad/mixed rows: 4


,id,quality_label,failure_tags,notes
7,r008,bad,[bad_top_1],Outlast has many jump scares; bad recommendation for this query despite investigation elements.
9,r010,mixed,[bad_top_1],Not really open-world exploration. Gorgeous game though.
14,r015,bad,[bad_top_1],Football Manager is not a racing game. Dirt 4 (rank 2) was better.
15,r016,bad,[bad_top_1],Bad recommendation for this query despite heavy investigation content.
